In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set the path to your shared folder
# This works for everyone because of the shortcut they made in Step 1!
project_path = '/content/drive/MyDrive/RU_AI_Campus_Team'

# 3. Move into the folder
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"Success! Working in: {os.getcwd()}")
else:
    print("Error: Could not find the project folder. Did you add the shortcut to your My Drive?")

In [ ]:
import pandas as pd

df = pd.read_csv("samhsa_filtered_data.csv")
df.head()

In [ ]:
df.info()

### User Input Simulation

This cell defines variables to simulate user input for filtering treatment facilities based on specific criteria.

In [ ]:
# Example user input variables
user_zip_code = '11203'
user_care_need = 'Mental Health'
user_setting = 'Outpatient'
user_age_group = 'Adults'
user_payment_preference = 'Medicaid'
query = 'therapy for adults in Brooklyn'

print(f"Simulated User Input:\n")
print(f"ZIP Code: {user_zip_code}")
print(f"Care Need: {user_care_need}")
print(f"Setting: {user_setting}")
print(f"Age Group: {user_age_group}")
print(f"Payment Preference: {user_payment_preference}")

In [ ]:
import pandas as pd
import re

# Clean helper
def contains_text(series, pattern):
    return series.fillna("").astype(str).str.contains(pattern, case=False, na=False)

# Normalize ZIP
df["zip"] = df["zip"].astype(str).str.zfill(5)

# Start with ZIP filter
filtered = df[df["zip"] == user_zip_code].copy()

# Care need filter
care = user_care_need.lower()
if care == "mental health":
    filtered = filtered[contains_text(filtered["type_of_care"], "mental health")]
elif care == "substance use":
    filtered = filtered[contains_text(filtered["type_of_care"], "substance use")]
elif care == "both":
    filtered = filtered[contains_text(filtered["type_of_care"], "co-occurring")]

# Setting filter
setting = user_setting.lower()
if setting == "outpatient":
    filtered = filtered[contains_text(filtered["service_setting"], "outpatient")]
elif setting == "residential":
    filtered = filtered[contains_text(filtered["service_setting"], "residential")]
elif setting == "inpatient":
    filtered = filtered[contains_text(filtered["service_setting"], "inpatient")]

# Age group filter
age = user_age_group.lower()
if age == "adults":
    filtered = filtered[contains_text(filtered["age_groups_accepted"], "adults")]
elif age == "children":
    filtered = filtered[contains_text(filtered["age_groups_accepted"], "children")]
elif age == "seniors":
    filtered = filtered[contains_text(filtered["age_groups_accepted"], "seniors")]

# Payment filter
payment = user_payment_preference.lower()
if payment == "medicaid":
    filtered = filtered[contains_text(filtered["payment_funding"], "medicaid")]
elif payment == "medicare":
    filtered = filtered[contains_text(filtered["payment_funding"], "medicare")]
elif payment == "private":
    filtered = filtered[contains_text(filtered["payment_funding"], "private health insurance")]
elif payment in ["payment assistance", "assistance", "sliding scale"]:
    filtered = filtered[
        contains_text(filtered["payment_assistance"], "sliding fee scale") |
        contains_text(filtered["payment_assistance"], "payment assistance")
    ]

print("Rows after filtering:", len(filtered))

cols_to_show = [
    "facility_name", "city", "zip", "phone",
    "type_of_care", "service_setting", "age_groups_accepted",
    "payment_funding", "payment_assistance"
]

filtered[cols_to_show].head(20)

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
import re
import pandas as pd

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[,/()\-]", " ", x)   # remove punctuation/slashes
    x = re.sub(r"\s+", " ", x).strip()
    return x

filtered["search_text"] = (
    filtered["facility_name"].apply(clean_text) + " " +
    filtered["city"].apply(clean_text) + " " +
    filtered["type_of_care"].apply(clean_text) + " " +
    filtered["service_setting"].apply(clean_text) + " " +
    filtered["treatment_approaches"].apply(clean_text) + " " +
    filtered["payment_funding"].apply(clean_text) + " " +
    filtered["payment_assistance"].apply(clean_text) + " " +
    filtered["age_groups_accepted"].apply(clean_text)
).str.replace(r"\s+", " ", regex=True).str.strip()

In [ ]:
filtered['search_text'].iloc[0]

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    filtered["search_text"].tolist(),
    show_progress_bar=True
)

In [ ]:
filtered["embedding"] = list(embeddings)

In [ ]:
query_embedding = model.encode([query])[0]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# convert embeddings to matrix
embedding_matrix = np.vstack(filtered["embedding"].values)

# compute similarity
scores = cosine_similarity([query_embedding], embedding_matrix)[0]

filtered["semantic_score"] = scores

In [ ]:
results = filtered.sort_values("semantic_score", ascending=False).head(10)

results[[
    "facility_name",
    "city",
    "semantic_score",
    "type_of_care",
    "service_setting"
]]

In [ ]:
def compute_rule_score(row, user_care_need, user_setting, user_age_group, user_payment):
    score = 0

    # Care type (strong)
    if user_care_need.lower() in str(row["type_of_care"]).lower():
        score += 40

    # Setting (strong)
    if user_setting.lower() in str(row["service_setting"]).lower():
        score += 30

    # Age (medium)
    if user_age_group.lower() in str(row["age_groups_accepted"]).lower():
        score += 15

    # Payment (medium)
    if user_payment.lower() in str(row["payment_funding"]).lower():
        score += 15

    # Payment assistance bonus
    if "sliding fee" in str(row["payment_assistance"]).lower():
        score += 5

    # Therapy richness bonus
    therapies = str(row["treatment_approaches"]).lower()
    if "cognitive behavioral therapy" in therapies:
        score += 5
    if "telemedicine" in therapies:
        score += 3

    return score

In [ ]:
filtered["rule_score"] = filtered.apply(
    lambda row: compute_rule_score(
        row,
        user_care_need,
        user_setting,
        user_age_group,
        user_payment_preference
    ),
    axis=1
)

In [ ]:
filtered["semantic_score_norm"] = filtered["semantic_score"] * 100
filtered["rule_score_norm"] = (filtered["rule_score"] / 113) * 100
filtered["final_score"] = (
    0.6 * filtered["semantic_score_norm"] +
    0.4 * filtered["rule_score_norm"]
)

In [ ]:
results = filtered.sort_values("final_score", ascending=False).head(10)

results[[
    "facility_name",
    "city",
    "semantic_score",
    "rule_score",
    "final_score",
    "type_of_care",
    "service_setting"
]]

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# 🔑 Get your API key from Colab secrets
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
def generate_explanation_with_gemini(reasons):
    if not reasons:
        return "Relevant based on available service information."

    # Updated prompt to encourage a more natural, single-sentence explanation
    prompt = f"""
    Combine the following facts into a single, cohesive, and concise sentence that explains why a behavioral health provider matches a user's needs. Do NOT add any new information and ensure the sentence flows naturally.

    Facts:
    {chr(10).join(['- ' + r for r in reasons])}

    The explanation should start directly with the reason, like "This facility offers X, provides Y..."
    """

    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        # Fallback in case of API error or model failure
        # Join reasons with a comma and an 'and' before the last one, then capitalize the first letter.
        if len(reasons) > 1:
            last_reason = reasons.pop()
            return f"{', '.join(reasons)} and {last_reason} .".capitalize()
        elif reasons:
            return f"{reasons[0]} .".capitalize()
        else:
            return "Relevant based on available service information."


In [ ]:
def get_reasons(row, user_care_need, user_setting, user_age_group, user_payment):
    reasons = []

    if user_care_need.lower() in str(row["type_of_care"]).lower():
        reasons.append("offers mental health services")

    if user_setting.lower() in str(row["service_setting"]).lower():
        reasons.append("provides outpatient care")

    if user_age_group.lower() in str(row["age_groups_accepted"]).lower():
        reasons.append("serves adults")

    if user_payment.lower() in str(row["payment_funding"]).lower():
        reasons.append("accepts Medicaid")

    if "sliding fee" in str(row["payment_assistance"]).lower():
        reasons.append("offers payment assistance")

    if "cognitive behavioral therapy" in str(row["treatment_approaches"]).lower():
        reasons.append("offers CBT")

    if "telemedicine" in str(row["treatment_approaches"]).lower():
        reasons.append("offers telehealth")

    return reasons

In [ ]:
top_results = results.head(10).copy()

top_results["reasons"] = top_results.apply(
    lambda row: get_reasons(
        row,
        user_care_need,
        user_setting,
        user_age_group,
        user_payment_preference
    ),
    axis=1
)

top_results["explanation"] = top_results["reasons"].apply(generate_explanation_with_gemini)

top_results[["facility_name", "explanation"]]

In [ ]:
top_results['explanation'].iloc[1]

In [ ]:
pd.options.display.max_columns = None
top_results

In [ ]:
top_results.columns

# **Experimentation Phase** 🤯

## Phase 1: Feature Engineering (The "Binary" Map)

Now, let's prepare the `ancillary_services` and `recovery_support` columns for binarization. These columns contain comma-separated strings of services. We need to parse these strings into lists of individual services and identify all unique services across the dataset.

We specifically chose `ancillary_services` and `recovery_support` because they provide the most granular and specific details about the *additional* programs and support a facility offers. While other columns like `type_of_care` (e.g., Mental Health) and `service_setting` (e.g., Outpatient) are great for initial filtering based on broad categories, these two columns reveal the unique 'care bundles' that truly differentiate facilities. Things like 'case management', 'housing services', or 'peer support' are found here, allowing us to build a more nuanced, tiered system for understanding facility offerings.

In [ ]:
import pandas as pd
import os

# Load the cleaned data into a new DataFrame for clustering tasks using the full path
# The 'project_path' variable is available from earlier executions in the notebook.
df_clustering = pd.read_csv("samhsa_filtered_data.csv")

print("DataFrame loaded successfully. Initial rows:")
display(df_clustering.head())

print("\nDataFrame information:")
df_clustering.info()

This code block is designed to extract and clean service information from two specific columns of the `df_clustering` DataFrame: `ancillary_services` and `recovery_support`. The goal is to prepare this data for a clustering task by creating a consolidated, standardized list of all services offered by each facility.

Here's a breakdown of what the code does:

1.  **`extract_and_clean_services(service_string)` function**:
    *   Takes a string (representing services from a column) as input.
    *   Checks if the input string is `NaN` (missing value); if so, it returns an empty list.
    *   Splits the service string into individual services using commas (`,`) or slashes (` / `) as delimiters.
    *   For each extracted service, it cleans up leading/trailing whitespace and converts the text to lowercase for consistency.
    *   Returns a list of cleaned, individual service strings.

2.  **Applying the function and combining services**:
    *   The `df_clustering.apply()` method is used to iterate through each row of the DataFrame.
    *   For each row, it calls `extract_and_clean_services` on both the `ancillary_services` and `recovery_support` columns.
    *   The results (two lists of cleaned services) are concatenated (`+`) to form a single list named `all_services_list`.
    *   This combined list is then assigned to a new column in `df_clustering` called `all_services_list`.

3.  **Displaying combined services**:
    *   A loop iterates through the `all_services_list` for the first few facilities.
    *   It prints the combined list of services for each of these facilities, providing a glimpse into the newly created feature.

In [ ]:
import re

def extract_and_clean_services(service_string):
    """Extracts and cleans individual services from a comma-separated string."""
    if pd.isna(service_string):
        return []
    # Split by comma or ' / ' and clean each service
    services = re.split(r',| / ', service_string)
    cleaned_services = [s.strip().lower() for s in services if s.strip()]
    return cleaned_services


# Apply the function to the relevant columns and combine them
df_clustering["all_services_list"] = df_clustering.apply(
    lambda row:
        extract_and_clean_services(row["ancillary_services"]) +
        extract_and_clean_services(row["recovery_support"]),
    axis=1
)

# Display the first few combined service lists
print("Combined services for the first few facilities:")
for i, services in enumerate(df_clustering["all_services_list"].head()):
    print(f"Facility {i}: {services}")

With the services extracted into lists, we can now use `MultiLabelBinarizer` to convert these lists into binary (0/1) features. Each unique service identified will become its own column.

### What is `MultiLabelBinarizer`?
`MultiLabelBinarizer` is a utility class from scikit-learn that transforms a list of lists of labels (or in our case, a list of lists of services) into a binary matrix. Each unique label found across all lists becomes a separate column in the matrix. If a particular facility (row) has a specific service (column), the corresponding cell in the matrix will be `1`; otherwise, it will be `0`.

This conversion is crucial for machine learning algorithms like K-Means, which require numerical input, and helps represent the presence or absence of specific services for each facility.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Initialize MultiLabelBinarizer
mlb = MultiLabelBinarizer()

# Fit and transform the 'all_services_list' column
service_features = mlb.fit_transform(df_clustering["all_services_list"])

# Create a DataFrame from the binary features
features_df = pd.DataFrame(service_features, columns=mlb.classes_, index=df_clustering.index)

# Concatenate the new feature DataFrame with the original df_clustering
df_clustering = pd.concat([df_clustering, features_df], axis=1)

print("DataFrame with new binary service features added. Head of the new features:")
display(df_clustering[mlb.classes_].head())

print(f"\nTotal number of unique services identified: {len(mlb.classes_)}")
print("Updated DataFrame info with new features:")
df_clustering.info(verbose=True, show_counts=True)

We have successfully completed Phase 1: Feature Engineering. The `df_clustering` DataFrame now includes numerous new binary columns, one for each unique `ancillary_services` and `recovery_support` item, representing whether a facility offers that specific service. This binary map is now ready for clustering.

## Phase 2: Finding the "Optimal K" (The Elbow Method)

Now that we have binarized features representing the services offered by each facility, we can apply K-Means clustering. The next step is to determine the optimal number of clusters, often denoted as 'K'. The Elbow Method is a common technique for this, where we run K-Means for a range of K values and plot the inertia (within-cluster sum-of-squares). The 'elbow' point in the plot, where the rate of decrease in inertia sharply changes, suggests the optimal K.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Extract only the binary service features for clustering
# These are the columns generated by MultiLabelBinarizer
features_for_clustering = df_clustering[mlb.classes_]

print(f"Shape of features for clustering: {features_for_clustering.shape}")
display(features_for_clustering.head())

In [ ]:
# Implement the Elbow Method
wcss = [] # Within-cluster sum of squares
k_range = range(1, 15) # Test K from 1 to 14

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init to suppress warning
    kmeans.fit(features_for_clustering)
    wcss.append(kmeans.inertia_)

print("WCSS calculation complete.")

In [ ]:
# Plot the Elbow Method results
plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.grid(True)
plt.xticks(k_range)
plt.show()

print("Please examine the plot above to identify the 'elbow' point, which suggests the optimal number of clusters.")

We have now completed **Phase 2: Finding the "Optimal K" (The Elbow Method)**. The elbow plot above shows how the WCSS (Within-Cluster Sum of Squares) decreases as the number of clusters increases. The 'elbow' in this plot, where the rate of decrease in WCSS slows down significantly, usually indicates the most appropriate number of clusters.

Based on this plot, you should choose the value of 'K' that represents the elbow. Once you have identified this, we can proceed to Phase 3 to train the K-Means model with the chosen K and label the clusters.

## Phase 3: Training and "Tier" Labeling

With **K=5** chosen as the optimal number of clusters, we can now train the K-Means model and assign each facility to its respective cluster. Following this, we will analyze the characteristics of each cluster to understand its unique service profile.

In [ ]:
# Based on the Elbow Method plot, we choose K=5 as the optimal number of clusters.
optimal_k = 5

print(f"Chosen optimal K for clustering: {optimal_k}")

In [ ]:
# Train K-Means model with the chosen optimal_k
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_clustering['cluster_label'] = kmeans_model.fit_predict(features_for_clustering)

print("K-Means clustering complete. Cluster labels assigned.")
display(df_clustering[['facility_name', 'cluster_label']].head())

In [ ]:
# Analyze cluster characteristics by calculating the mean of each feature for each cluster
cluster_feature_means = df_clustering.groupby('cluster_label')[mlb.classes_].mean()

print("Average service presence per cluster:")
display(cluster_feature_means)

We have now trained the K-Means model with `K=5` and calculated the average presence of each service within these five clusters. The `cluster_feature_means` DataFrame shows us the distinct profile of services offered by facilities in each cluster.

**Next, we need your expertise to interpret these clusters!**

Based on the `cluster_feature_means` output, please analyze the service profiles of each cluster and suggest a meaningful, descriptive **"tier" name** for each. For example, if a cluster has a high average for many services, you might call it a "Comprehensive Hub." If another specializes in a few, it might be a "Specialty Boutique."

Let me know your suggested names for each of the 5 clusters!

### Interpreting and Naming the Clusters

Based on the `cluster_feature_means` DataFrame, we can assign descriptive names to each cluster to better understand the types of facilities they represent. These names will serve as 'tiers' for the service offerings.

In [ ]:
# Define the mapping of cluster labels to tier names
cluster_tier_names = {
    0: 'Comprehensive Support Hub',
    1: 'Peer & Community-Led Services',
    2: 'Specialized Chronic Care',
    3: 'Structured Clinical & Family Support',
    4: 'Essential Support Services'
}

# Map the cluster labels in df_clustering to the new tier names
df_clustering['tier_name'] = df_clustering['cluster_label'].map(cluster_tier_names)

print("Cluster labels have been mapped to descriptive tier names.")
display(df_clustering[['facility_name', 'cluster_label', 'tier_name']].head())

In [ ]:
df_clustering.head()

In [ ]:
df_clustering.info()

In [ ]:
# Export the final clustered DataFrame to a CSV file
df_clustering.to_csv('clustered_dataframe_final.csv', index=False)
print("DataFrame 'df_clustering' exported to 'clustered_dataframe_final.csv' successfully.")